# Preparación de datos para K-Means

Este notebook prepara el dataset limpio de consumo eléctrico para su posterior
procesamiento mediante el algoritmo K-Means implementado en Go.

A partir del archivo `hhblock_dataset_limpio.csv`, se seleccionan las 48 variables
correspondientes a los intervalos de media hora del día (`hh_0` a `hh_47`),
se realizan validaciones adicionales y se normalizan los perfiles diarios de consumo.

Como resultado se generan dos archivos:

- `kmeans_input.csv`: contiene únicamente las características numéricas utilizadas por K-Means.
- `kmeans_metadata.csv`: conserva el identificador del hogar y la fecha para relacionar posteriormente los clusters obtenidos.

In [2]:
import pandas as pd
import numpy as np

## 1. Carga del dataset limpio

Se utiliza como entrada el dataset resultante de la etapa de limpieza realizada
previamente en la PC1.

In [3]:
ruta = "./data/hhblock_dataset_limpio.csv"

df = pd.read_csv(ruta)

print("Dimensiones:", df.shape)
df.head()

Dimensiones: (3469352, 50)


,LCLid,day,hh_0,hh_1,hh_2,hh_3,hh_4,hh_5,hh_6,hh_7,...,hh_38,hh_39,hh_40,hh_41,hh_42,hh_43,hh_44,hh_45,hh_46,hh_47
0,MAC000002,2012-10-13,0.263,0.269,0.275,0.256,0.211,0.136,0.161,0.119,...,0.918,0.278,0.267,0.239,0.230,0.233,0.235,0.188,0.259,0.250
1,MAC000002,2012-10-14,0.262,0.166,0.226,0.088,0.126,0.082,0.123,0.083,...,1.075,0.956,0.821,0.745,0.712,0.511,0.231,0.210,0.278,0.159
2,MAC000002,2012-10-15,0.192,0.097,0.141,0.083,0.132,0.070,0.130,0.074,...,1.164,0.249,0.225,0.258,0.260,0.334,0.299,0.236,0.241,0.237
3,MAC000002,2012-10-16,0.237,0.237,0.193,0.118,0.098,0.107,0.094,0.109,...,0.966,0.172,0.192,0.228,0.203,0.211,0.188,0.213,0.157,0.202
4,MAC000002,2012-10-17,0.157,0.211,0.155,0.169,0.101,0.117,0.084,0.118,...,0.223,0.075,0.230,0.208,0.265,0.377,0.327,0.277,0.288,0.256


## 2. Selección de características

Cada registro representa el perfil de consumo eléctrico de un hogar durante un día.

Para el algoritmo K-Means se utilizarán únicamente las 48 columnas correspondientes
a los intervalos de media hora (`hh_0` a `hh_47`).

Las variables `LCLid` y `day` no participan en el cálculo de distancias, ya que funcionan
como identificadores del registro.

In [4]:
cols_hh = [f"hh_{i}" for i in range(48)]

X = df[cols_hh].copy()

print("Características utilizadas:", len(cols_hh))
print("Dimensiones de X:", X.shape)

X.head()

Características utilizadas: 48
Dimensiones de X: (3469352, 48)


,hh_0,hh_1,hh_2,hh_3,hh_4,hh_5,hh_6,hh_7,hh_8,hh_9,...,hh_38,hh_39,hh_40,hh_41,hh_42,hh_43,hh_44,hh_45,hh_46,hh_47
0,0.263,0.269,0.275,0.256,0.211,0.136,0.161,0.119,0.167,0.109,...,0.918,0.278,0.267,0.239,0.230,0.233,0.235,0.188,0.259,0.250
1,0.262,0.166,0.226,0.088,0.126,0.082,0.123,0.083,0.120,0.079,...,1.075,0.956,0.821,0.745,0.712,0.511,0.231,0.210,0.278,0.159
2,0.192,0.097,0.141,0.083,0.132,0.070,0.130,0.074,0.124,0.078,...,1.164,0.249,0.225,0.258,0.260,0.334,0.299,0.236,0.241,0.237
3,0.237,0.237,0.193,0.118,0.098,0.107,0.094,0.109,0.091,0.105,...,0.966,0.172,0.192,0.228,0.203,0.211,0.188,0.213,0.157,0.202
4,0.157,0.211,0.155,0.169,0.101,0.117,0.084,0.118,0.080,0.119,...,0.223,0.075,0.230,0.208,0.265,0.377,0.327,0.277,0.288,0.256


## 3. Validaciones previas al modelado

Antes de aplicar K-Means se verifica que las variables de consumo no presenten
valores nulos, infinitos, negativos o perfiles cuyo consumo diario total sea cero.

In [5]:
nulos = X.isnull().sum().sum()

infinitos = np.isinf(X.to_numpy()).sum()

negativos = (X < 0).sum().sum()

consumo_total = X.sum(axis=1)
perfiles_cero = (consumo_total == 0).sum()

print("Valores nulos:", nulos)
print("Valores infinitos:", infinitos)
print("Valores negativos:", negativos)
print("Perfiles con consumo total igual a 0:", perfiles_cero)

Valores nulos: 0
Valores infinitos: 0
Valores negativos: 0
Perfiles con consumo total igual a 0: 14877


## 4. Tratamiento de perfiles sin consumo

Durante la validación se identificaron 14,877 registros cuyo consumo total diario es igual a cero, lo que representa aproximadamente el 0.43% del dataset.

Estos perfiles fueron excluidos de la entrada utilizada por K-Means, debido a que no contienen información sobre la distribución del consumo eléctrico durante el día y no pueden ser normalizados mediante la división entre el consumo total.

Después de esta exclusión, se conservaron únicamente los perfiles con consumo total mayor a cero para las siguientes etapas de preparación y modelado.

In [6]:
mask_validos = consumo_total > 0

print("Registros originales:", len(X))
print("Registros válidos:", mask_validos.sum())
print("Registros excluidos:", (~mask_validos).sum())

X = X.loc[mask_validos].reset_index(drop=True)

metadata = df.loc[
    mask_validos,
    ["LCLid", "day"]
].reset_index(drop=True)

print("Dimensiones de X:", X.shape)
print("Dimensiones de metadata:", metadata.shape)

Registros originales: 3469352
Registros válidos: 3454475
Registros excluidos: 14877
Dimensiones de X: (3454475, 48)
Dimensiones de metadata: (3454475, 2)


In [7]:
# Normalización de perfiles diarios

totales = X.sum(axis=1)

X_normalizado = X.div(totales, axis=0)

X_normalizado.head()

,hh_0,hh_1,hh_2,hh_3,hh_4,hh_5,hh_6,hh_7,hh_8,hh_9,...,hh_38,hh_39,hh_40,hh_41,hh_42,hh_43,hh_44,hh_45,hh_46,hh_47
0,0.023721,0.024263,0.024804,0.023090,0.019031,0.012267,0.014522,0.010733,0.015063,0.009831,...,0.082800,0.025074,0.024082,0.021557,0.020745,0.021016,0.021196,0.016957,0.023361,0.022549
1,0.019814,0.012554,0.017091,0.006655,0.009529,0.006201,0.009302,0.006277,0.009075,0.005974,...,0.081298,0.072298,0.062089,0.056341,0.053846,0.038645,0.017470,0.015881,0.021024,0.012025
2,0.018719,0.009457,0.013747,0.008092,0.012869,0.006825,0.012674,0.007215,0.012089,0.007605,...,0.113483,0.024276,0.021936,0.025154,0.025349,0.032563,0.029151,0.023009,0.023496,0.023106
3,0.024260,0.024260,0.019756,0.012079,0.010032,0.010953,0.009622,0.011158,0.009315,0.010748,...,0.098884,0.017607,0.019654,0.023339,0.020780,0.021599,0.019245,0.021804,0.016071,0.020678
4,0.014424,0.019384,0.014240,0.015526,0.009279,0.010749,0.007717,0.010841,0.007350,0.010932,...,0.020487,0.006890,0.021130,0.019109,0.024345,0.034635,0.030041,0.025448,0.026458,0.023519


In [8]:
suma_perfiles = X_normalizado.sum(axis=1)

print("Mínimo:", suma_perfiles.min())
print("Máximo:", suma_perfiles.max())
print("Promedio:", suma_perfiles.mean())

print(
    "Perfiles correctamente normalizados:",
    np.allclose(suma_perfiles.to_numpy(), 1.0)
)

Mínimo: 0.9999999999999976
Máximo: 1.000000000000003
Promedio: 1.0
Perfiles correctamente normalizados: True


In [9]:
X_normalizado = X_normalizado.astype("float32")

print(X_normalizado.dtypes.value_counts())

float32    48
Name: count, dtype: int64


In [10]:
print("Dimensiones finales:", X_normalizado.shape)

print("Nulos:", X_normalizado.isnull().sum().sum())
print("Infinitos:", np.isinf(X_normalizado.to_numpy()).sum())
print("Negativos:", (X_normalizado < 0).sum().sum())

Dimensiones finales: (3454475, 48)
Nulos: 0
Infinitos: 0
Negativos: 0


In [11]:
X_normalizado.to_csv(
    "./data/kmeans_input.csv",
    index=False
)

metadata.to_csv(
    "./data/kmeans_metadata.csv",
    index=False
)

print("kmeans_input.csv:", X_normalizado.shape)
print("kmeans_metadata.csv:", metadata.shape)

kmeans_input.csv: (3454475, 48)
kmeans_metadata.csv: (3454475, 2)
